# Agriculture Data Cleaning: USGS County Nutrient Inputs from Fertilizer (Falcone 2020)

Cleans the USGS Falcone (2020) county-level **fertilizer** nitrogen (N) and phosphorus (P) workbook into one tidy long table: one row per `(county_fips, year, nutrient, source)`.

**Input:**  `data/tabular/01_raw/agriculture/N-P_from_fertilizer_1950-2017-july23-2020.xlsx`
**Output:** `data/tabular/02_clean/agriculture/np-fertilizer-clean.csv`

**What's in it** — county-level kilograms of N and P applied as fertilizer for ~five-year periods, 1950-2017. The raw workbook is national; this project is an Iowa water-quality study, so the output is filtered to Iowa's 99 counties (state FIPS 19). The workbook splits the record across three sheets that cover *different* eras and so become a `source` column:

- `total` — farm + non-farm **combined**, periods **1950-1982** (the pre-split era; Alexander & Smith 1990).
- `farm` / `nonfarm` — the farm vs. non-farm split available only for periods **1987-2017** (Brakebill & Gronberg 2017; Falcone 2020).

We keep the sources exactly as published (no synthesised totals) and reshape each wide `<prefix>fert{N,P}-kg-{year}` block into long rows.

**Pipeline**
1. **Load** the three data sheets. 2. **Guard** the schema (row count, unique county FIPS, expected year sets). 3. **Melt** each sheet to long, parsing `nutrient` and `year` from the column name. 4. **Concat**, build FIPS keys. 5. **Key**, check, save.

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "agriculture"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "agriculture"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# Project scope: this is an Iowa water-quality study, so we narrow the national
# workbook to Iowa (state FIPS 19 / postal "IA", 99 counties) for the output.
# Schema guards still run against the full national pull before we filter.
STATE = "IA"
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)


def add_fips(df: pd.DataFrame) -> pd.DataFrame:
    """Zero-pad STCOFIPS into 5-digit county_fips and derive 2-digit state_fips."""
    df = df.copy()
    df["county_fips"] = df["STCOFIPS"].astype(int).astype(str).str.zfill(5)
    df["state_fips"] = df["county_fips"].str[:2]
    return df.rename(columns={"CountyName": "county_name", "State": "state"})


Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction
Raw dir:   /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/01_raw/agriculture
Clean dir: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/agriculture


## Step 1 - Load the three data sheets

In [2]:
RAW_FILE = "N-P_from_fertilizer_1950-2017-july23-2020.xlsx"
xl = pd.ExcelFile(RAW_DIR / RAW_FILE)
assert {"total", "farm", "nonfarm", "notes"} <= set(xl.sheet_names), xl.sheet_names

sheets = {name: xl.parse(name) for name in ("total", "farm", "nonfarm")}
for name, d in sheets.items():
    print(f"{name:8s} {d.shape[0]:,} rows x {d.shape[1]} cols")
sheets["total"].head(3)

total    3,066 rows x 20 cols
farm     3,066 rows x 18 cols
nonfarm  3,066 rows x 18 cols


,STCOFIPS,fips-int,CountyName,State,tot-fertN-kg-1950,tot-fertN-kg-1954,tot-fertN-kg-1959,tot-fertN-kg-1964,tot-fertN-kg-1969,tot-fertN-kg-1974,tot-fertN-kg-1978,tot-fertN-kg-1982,tot-fertP-kg-1950,tot-fertP-kg-1954,tot-fertP-kg-1959,tot-fertP-kg-1964,tot-fertP-kg-1969,tot-fertP-kg-1974,tot-fertP-kg-1978,tot-fertP-kg-1982
0,1001,1001,Autauga,AL,962208.0,1309789.0,1365294.0,1680112.0,2235628.0,2833962.0,2500111.0,2345498.0,7.678911e+05,721978.125,545900.625,7.228443e+05,748171.500,8.164091e+05,736223.375,605069.875
1,1003,1003,Baldwin,AL,2944313.0,4007896.0,4177737.0,5141065.0,6840920.0,8671794.0,6472689.0,5843222.0,2.349712e+06,2209221.000,1670431.000,2.211870e+06,2289371.000,2.498175e+06,1906053.000,1507381.000
2,1005,1005,Barbour,AL,1013780.0,1379991.0,1438471.0,1770162.0,2355453.0,2985856.0,2625201.0,2125210.0,8.090483e+05,760674.500,575159.625,7.615870e+05,788271.875,8.601668e+05,773059.125,548242.000


## Step 2 - Schema guards

Every sheet should carry one row per county (3,066 conterminous-U.S. counties, unique FIPS) and the era-specific year set. Guard against a future re-pull silently changing scope or the value-column naming.

In [3]:
ID_COLS = ["STCOFIPS", "fips-int", "CountyName", "State"]

# (regex with named groups, expected year set) per source sheet
SOURCE_PATTERNS = {
    "total":   (re.compile(r"^tot-fert(?P<nutrient>[NP])-kg-(?P<year>\d{4})$"),
                {1950, 1954, 1959, 1964, 1969, 1974, 1978, 1982}),
    "farm":    (re.compile(r"^farmfert(?P<nutrient>[NP])-kg-(?P<year>\d{4})$"),
                {1987, 1992, 1997, 2002, 2007, 2012, 2017}),
    "nonfarm": (re.compile(r"^nonffert(?P<nutrient>[NP])-kg-(?P<year>\d{4})$"),
                {1987, 1992, 1997, 2002, 2007, 2012, 2017}),
}

for name, d in sheets.items():
    assert len(d) == 3066, f"{name}: expected 3066 rows, got {len(d)}"
    assert not d["STCOFIPS"].duplicated().any(), f"{name}: duplicate county FIPS"
    pat, expected_years = SOURCE_PATTERNS[name]
    val_cols = [c for c in d.columns if pat.match(str(c))]
    unmatched = [c for c in d.columns if c not in ID_COLS
                 and "Unnamed" not in str(c) and not pat.match(str(c))]
    assert not unmatched, f"{name}: unexpected columns {unmatched}"
    years = {int(pat.match(c)["year"]) for c in val_cols}
    assert years == expected_years, f"{name}: years {years} != {expected_years}"
print("Schema guards passed: 3,066 counties each, expected year sets, all value cols parse.")

Schema guards passed: 3,066 counties each, expected year sets, all value cols parse.


## Step 3 - Melt each sheet to long

Each value column is `<prefix>fert{N,P}-kg-{year}`. We melt to one row per `(county, nutrient, year)` and tag the originating sheet as `source`.

In [4]:
def tidy_fert(df: pd.DataFrame, source: str, pat: re.Pattern) -> pd.DataFrame:
    val_cols = [c for c in df.columns if pat.match(str(c))]
    long = df.melt(
        id_vars=["STCOFIPS", "CountyName", "State"],
        value_vars=val_cols, var_name="col", value_name="value_kg",
    )
    meta = long["col"].str.extract(pat)
    long["nutrient"] = meta["nutrient"]
    long["year"] = meta["year"].astype(int)
    long["source"] = source
    return long.drop(columns="col")

long = pd.concat(
    [tidy_fert(sheets[name], name, SOURCE_PATTERNS[name][0]) for name in sheets],
    ignore_index=True,
)
print(f"Melted to {len(long):,} long rows")
print(long.groupby("source")["year"].agg(["min", "max", "nunique"]))

Melted to 134,904 long rows
          min   max  nunique
source                      
farm     1987  2017        7
nonfarm  1987  2017        7
total    1950  1982        8


## Step 4 - Build FIPS keys, filter to Iowa, select and order columns

In [5]:
long = add_fips(long)
long["value_kg"] = pd.to_numeric(long["value_kg"], errors="coerce")

long = long[long["state"] == STATE].copy()  # project scope: Iowa only
assert long["county_fips"].str[:2].eq("19").all(), "non-Iowa rows after filter"
print(f"Filtered to {STATE}: {long.county_fips.nunique()} counties")

OUTPUT_COLS = ["state_fips", "county_fips", "county_name", "state",
               "year", "nutrient", "source", "value_kg"]
clean = long[OUTPUT_COLS].sort_values(
    ["county_fips", "source", "nutrient", "year"]
).reset_index(drop=True)

KEY = ["county_fips", "year", "nutrient", "source"]
dupes = clean.duplicated(KEY).sum()
assert dupes == 0, f"{dupes} duplicate key rows!"
print(f"Key is unique across {len(clean):,} rows.")
clean.head()

Filtered to IA: 99 counties
Key is unique across 4,356 rows.


,state_fips,county_fips,county_name,state,year,nutrient,source,value_kg
0,19,19001,Adair,IA,1987,N,farm,7027411.0
1,19,19001,Adair,IA,1992,N,farm,7244305.0
2,19,19001,Adair,IA,1997,N,farm,6987657.0
3,19,19001,Adair,IA,2002,N,farm,6456246.0
4,19,19001,Adair,IA,2007,N,farm,7282474.0


## Step 5 - Sanity check

In [6]:
print(f"Rows: {len(clean):,}  |  counties: {clean.county_fips.nunique():,}  |  "
      f"years: {clean.year.min()}-{clean.year.max()}")
print(f"value_kg missing: {clean.value_kg.isna().sum():,} (non-disclosed farm values)\n")
print("Iowa (FIPS 19) total fertilizer-N by source-era, kg:")
ia = clean[(clean.state == "IA") & (clean.nutrient == "N")]
print(ia.groupby(["source", "year"])["value_kg"].sum().round(0).to_string())

Rows: 4,356  |  counties: 99  |  years: 1950-2017
value_kg missing: 0 (non-disclosed farm values)

Iowa (FIPS 19) total fertilizer-N by source-era, kg:
source   year
farm     1987    7.716860e+08
         1992    8.760184e+08
         1997    8.844379e+08
         2002    7.967242e+08
         2007    1.084989e+09
         2012    1.218486e+09
         2017    1.123074e+09
nonfarm  1987    1.717837e+06
         1992    2.881269e+06
         1997    5.945981e+06
         2002    7.313370e+06
         2007    6.578615e+06
         2012    6.934938e+06
         2017    6.304210e+06
total    1950    1.247621e+07
         1954    6.904227e+07
         1959    8.502415e+07
         1964    2.477908e+08
         1969    5.050855e+08
         1974    6.808517e+08
         1978    8.354506e+08
         1982    9.513346e+08


## Step 6 - Save

In [7]:
out_file = CLEAN_DIR / "np-fertilizer-clean.csv"
clean.to_csv(out_file, index=False)
print(f"Saved {len(clean):,} rows -> {out_file}")

Saved 4,356 rows -> /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/agriculture/np-fertilizer-clean.csv
